<a href="https://colab.research.google.com/github/NazHub1993/ML_Notebooks/blob/main/Hyper%20Parameter%20Tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [146]:
import pandas as pd
df=pd.read_csv('iris.csv')
df.head()

,sepal_length,sepal_width,petal_length,petal_width,species
0,5.1,3.5,1.4,0.2,setosa
1,4.9,3.0,1.4,0.2,setosa
2,4.7,3.2,1.3,0.2,setosa
3,4.6,3.1,1.5,0.2,setosa
4,5.0,3.6,1.4,0.2,setosa


In [147]:
df.groupby('species').describe()

sepal_length                                              \
                  count   mean       std  min    25%  50%  75%  max   
species                                                               
setosa             50.0  5.006  0.352490  4.3  4.800  5.0  5.2  5.8   
versicolor         50.0  5.936  0.516171  4.9  5.600  5.9  6.3  7.0   
virginica          50.0  6.588  0.635880  4.9  6.225  6.5  6.9  7.9   

           sepal_width         ... petal_length      petal_width         \
                 count   mean  ...          75%  max       count   mean   
species                        ...                                        
setosa            50.0  3.418  ...        1.575  1.9        50.0  0.244   
versicolor        50.0  2.770  ...        4.600  5.1        50.0  1.326   
virginica         50.0  2.974  ...        5.875  6.9        50.0  2.026   

                                               
                 std  min  25%  50%  75%  max  
species                                        
setosa      0.107210  0.1  0.2  0.2  0.3  0.6  
versicolor  0.197753  1.0  1.2  1.3  1.5  1.8  
virginica   0.274650  1.4  1.8  2.0  2.3  2.5  

[3 rows x 32 columns]

In [148]:
df.columns[df.isna().any()]

Index([], dtype='object')

In [149]:
target=df.species
inputs=df.drop('species',axis='columns')

In [150]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(inputs,target,test_size=0.20)

In [151]:
from sklearn import svm
model=svm.SVC(kernel='rbf',C=30,gamma='auto')
model.fit(x_train,y_train)
model.score(x_test,y_test)


0.9666666666666667

#But how do I know that these parameters were proper kernel='rbf',C=30 ???

#Again How do I know that this test or training set was correct?

In [152]:
from sklearn.model_selection import cross_val_score
cross_val_score(svm.SVC(kernel='linear',C=10,gamma='auto'),inputs,target,cv=5)

array([1.        , 1.        , 0.9       , 0.96666667, 1.        ])

In [153]:
cross_val_score(svm.SVC(kernel='rbf',C=50,gamma='auto'),inputs,target,cv=5)

array([1.        , 0.96666667, 0.9       , 0.93333333, 1.        ])

In [154]:
import numpy as np
gamma='auto'
kernel=['rbf','linear']
C=[10,20,30]

for kval in kernel:
    for cval in C:
        cv_scores=cross_val_score(svm.SVC(kernel=kval,C=cval,gamma='auto'),inputs,target,cv=5)
        avg_score=np.average(cv_scores)
        print(kval,cval,avg_score)


rbf 10 0.9800000000000001
rbf 20 0.9666666666666668
rbf 30 0.96
linear 10 0.9733333333333334
linear 20 0.9666666666666666
linear 30 0.96


#But it is very tiring to do this with for loop . So we do with Grid Search CV

In [155]:
from sklearn.model_selection import GridSearchCV
clf=GridSearchCV(svm.SVC(gamma='auto'),
                 {
                     'kernel':['rbf','linear'],
                     'C':[10,20,30,50]
                 }
                 ,cv=5,return_train_score=False)
clf.fit(inputs,target)
clf.cv_results_



{'mean_fit_time': array([0.00615849, 0.00399532, 0.00415096, 0.00409112, 0.00413661,
        0.00427713, 0.00406981, 0.00359874]),
 'std_fit_time': array([2.22848068e-03, 1.62504998e-04, 7.30399666e-05, 3.23626629e-04,
        1.87480728e-04, 5.57060112e-04, 3.37069000e-04, 1.35657062e-04]),
 'mean_score_time': array([0.00500426, 0.00330977, 0.00310526, 0.00285325, 0.00300779,
        0.00353708, 0.00306973, 0.00271955]),
 'std_score_time': array([1.72225176e-03, 3.99509652e-04, 9.98950061e-05, 5.14804403e-05,
        1.35153569e-04, 9.66304137e-04, 4.05544007e-04, 2.88033327e-05]),
 'param_C': masked_array(data=[10, 10, 20, 20, 30, 30, 50, 50],
              mask=[False, False, False, False, False, False, False, False],
        fill_value=999999),
 'param_kernel': masked_array(data=['rbf', 'linear', 'rbf', 'linear', 'rbf', 'linear',
                    'rbf', 'linear'],
              mask=[False, False, False, False, False, False, False, False],
        fill_value=np.str_('?'),
      

In [156]:
cf=pd.DataFrame(clf.cv_results_)
cf

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_C,param_kernel,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,0.006158,0.002228,0.005004,0.001722,10,rbf,"{'C': 10, 'kernel': 'rbf'}",0.966667,1.000000,0.966667,0.966667,1.0,0.980000,0.016330,1
1,0.003995,0.000163,0.003310,0.000400,10,linear,"{'C': 10, 'kernel': 'linear'}",1.000000,1.000000,0.900000,0.966667,1.0,0.973333,0.038873,2
2,0.004151,0.000073,0.003105,0.000100,20,rbf,"{'C': 20, 'kernel': 'rbf'}",0.966667,1.000000,0.900000,0.966667,1.0,0.966667,0.036515,3
3,0.004091,0.000324,0.002853,0.000051,20,linear,"{'C': 20, 'kernel': 'linear'}",1.000000,1.000000,0.900000,0.933333,1.0,0.966667,0.042164,4
4,0.004137,0.000187,0.003008,0.000135,30,rbf,"{'C': 30, 'kernel': 'rbf'}",0.966667,1.000000,0.900000,0.933333,1.0,0.960000,0.038873,6
5,0.004277,0.000557,0.003537,0.000966,30,linear,"{'C': 30, 'kernel': 'linear'}",1.000000,1.000000,0.900000,0.900000,1.0,0.960000,0.048990,6
6,0.004070,0.000337,0.003070,0.000406,50,rbf,"{'C': 50, 'kernel': 'rbf'}",1.000000,0.966667,0.900000,0.933333,1.0,0.960000,0.038873,6
7,0.003599,0.000136,0.002720,0.000029,50,linear,"{'C': 50, 'kernel': 'linear'}",1.000000,1.000000,0.900000,0.933333,1.0,0.966667,0.042164,4


In [157]:
cf[['param_kernel','param_C','mean_test_score']]

,param_kernel,param_C,mean_test_score
0,rbf,10,0.980000
1,linear,10,0.973333
2,rbf,20,0.966667
3,linear,20,0.966667
4,rbf,30,0.960000
5,linear,30,0.960000
6,rbf,50,0.960000
7,linear,50,0.966667


In [158]:
dir(clf)

['__abstractmethods__',
 '__annotations__',
 '__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__setstate__',
 '__sizeof__',
 '__sklearn_clone__',
 '__sklearn_tags__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_abc_impl',
 '_build_request_for_signature',
 '_check_feature_names',
 '_check_n_features',
 '_check_refit_for_multimetric',
 '_doc_link_module',
 '_doc_link_template',
 '_doc_link_url_param_generator',
 '_estimator_type',
 '_format_results',
 '_get_default_requests',
 '_get_doc_link',
 '_get_metadata_request',
 '_get_param_names',
 '_get_routed_params_for_fit',
 '_get_scorers',
 '_get_tags',
 '_more_tags',
 '_parameter_constraints',
 '_repr_html_',
 '_repr_html_inner',
 '_repr_mimebundle_',
 '_run_sea

In [159]:
clf.best_params_

{'C': 10, 'kernel': 'rbf'}

#This is how I get the best parameters

##Now how do I get the best model?

In [160]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
model_params={
    'svm':{
        'model':svm.SVC(gamma='auto'),
        'params':{
            'kernel':['rbf','linear'],
            'C':[10,20,30,50]
        }
    },
    'random_forest':{
        'model':RandomForestClassifier(),
        'params':{
            'n_estimators':[1,5,10]
        }
    },
    'logistic_regression':{
        'model':LogisticRegression(solver='liblinear',multi_class='auto'), # Changed solver to 'lbfgs'
        'params':{
            'C':[1,5,10]
        }
    }

}

In [161]:
scores=[]
for model_name,mp in model_params.items():
  clf=GridSearchCV(mp['model'],mp['params'],cv=5,return_train_score=False)
  clf.fit(inputs,target)
  scores.append({
      'model':model_name,
      'best_score':clf.best_score_,
      'best_params':clf.best_params_
  })
print(scores)

[{'model': 'svm', 'best_score': np.float64(0.9800000000000001), 'best_params': {'C': 10, 'kernel': 'rbf'}}, {'model': 'random_forest', 'best_score': np.float64(0.9533333333333334), 'best_params': {'n_estimators': 10}}, {'model': 'logistic_regression', 'best_score': np.float64(0.9666666666666668), 'best_params': {'C': 5}}]


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and wi